In [15]:
import pandas as pd

arome = pd.read_csv("../data/cleaned/Arome_clean_final.csv", dtype={"id": str})
metar = pd.read_csv("../data/raw/METAR_2021_2025.csv", dtype={"station": str}, low_memory=False)
mapping = pd.read_csv("../data/cleaned/isd_history_station_check.csv", dtype=str)

arome["datetime"] = pd.to_datetime(arome["datetime"].astype(str), format="%Y%m%d%H")

metar["valid"] = pd.to_datetime(metar["valid"])
metar["datetime"] = metar["valid"].dt.floor("h")

icao_to_wmo = dict(zip(mapping["icao"], mapping["id"]))
metar["wmo_id"] = metar["station"].map(icao_to_wmo)

In [16]:
print(arome["id"].nunique())
print(metar["wmo_id"].nunique())
print(sorted(arome["id"].unique()))
print(sorted(metar["wmo_id"].dropna().unique()))

26
20
['60033', '60060', '60096', '60101', '60107', '60115', '60120', '60135', '60136', '60141', '60150', '60155', '60156', '60160', '60191', '60200', '60210', '60220', '60230', '60250', '60252', '60265', '60280', '60285', '60338', '60340']
['60033', '60101', '60107', '60115', '60135', '60141', '60150', '60155', '60156', '60191', '60200', '60210', '60220', '60230', '60252', '60265', '60280', '60285', '60338', '60340']


In [17]:
print(arome["datetime"].min(), arome["datetime"].max())
print(metar["datetime"].min(), metar["datetime"].max())

2019-01-01 21:00:00 2025-12-31 23:00:00
2021-01-01 00:00:00 2025-12-30 23:00:00


In [19]:
set(arome["id"]) - set(metar["wmo_id"].dropna())

{'60060', '60096', '60120', '60136', '60160', '60250'}

In [20]:
mapping[mapping["id"].isin(["60060","60096","60120","60136","60160","60250"])]

,id,found,coord_match,icao,station_name,lat_arome,lon_arome,lat_isd,lon_isd,begin,end
1,60060,True,True,GMMF,SIDI IFNI,29.35,-10.175,29.369,-10.18,19500310,20250824
2,60096,True,True,GSVO,VILLA CISNEROS(MIL),23.7,-15.85,23.7,-15.867,19310103,20250404
6,60120,True,True,GMMP,KENITRA (RMAFB),34.3,-6.6,34.3,-6.6,19490122,20250824
8,60136,True,True,GMSL,SIDI SLIMANE,34.225,-6.05,34.233,-6.05,19510901,20210612
13,60160,True,True,GMFI,IFRANE,33.5,-5.15,33.505,-5.153,19490209,19971221
19,60250,True,True,GMAA,INEZGANE,30.375,-9.55,30.381,-9.546,19450615,20250824


In [21]:
missing_icao = ["GMMF", "GSVO", "GMMP", "GMSL", "GMFI", "GMAA"]

metar[metar["station"].isin(missing_icao)]["station"].value_counts()

Series([], Name: count, dtype: int64)

In [22]:
sorted(metar["station"].unique())

['GEML',
 'GMAD',
 'GMAG',
 'GMAT',
 'GMFB',
 'GMFF',
 'GMFK',
 'GMFM',
 'GMFO',
 'GMMC',
 'GMMD',
 'GMME',
 'GMMI',
 'GMML',
 'GMMN',
 'GMMW',
 'GMMX',
 'GMMZ',
 'GMTA',
 'GMTT']

In [23]:
len(metar["station"].unique())

20

In [26]:
mapping[["icao", "id"]]

,icao,id
0,GMML,60033
1,GMMF,60060
2,GSVO,60096
3,GMTT,60101
4,GMTA,60107
5,GMFO,60115
6,GMMP,60120
7,GMME,60135
8,GMSL,60136
9,GMFF,60141


In [1]:
import pandas as pd

df = pd.read_csv(
    "../data/raw/NOAA_ISD_2021_2025.csv",
    usecols=["icao", "REM"],
    dtype=str,
)

empty = df["REM"].isna() | (df["REM"].str.strip() == "")
print(f"REM vide : {empty.sum():,} / {len(df):,} ({empty.mean()*100:.2f}%)")

print(df.loc[empty, "icao"].value_counts())

REM vide : 0 / 1,324,764 (0.00%)
Series([], Name: count, dtype: int64)


In [3]:
import pandas as pd

df = pd.read_csv("../data/raw/METAR_2021_2025.csv", parse_dates=["date"])
mism = df[df["gust_mismatch"]]

only_oc1 = mism[mism["has_gust"] == 1]       # OC1 dit rafale, avwx non
only_avwx = mism[mism["has_gust"] == 0]      # avwx dit rafale, OC1 non

print(f"OC1 seul (pas dans le texte METAR): {len(only_oc1):,}")
print(f"avwx seul (pas dans OC1): {len(only_avwx):,}")

# Hypothèse seuil OACI 10kt (~5.14 m/s) : écart rafale - vent moyen
delta = only_oc1["gust_speed_ms"] - only_oc1["wind_speed_ms"]
print(delta.describe())
print(f"Sous le seuil des 10kt (5.14 m/s): {(delta < 5.14).mean() * 100:.1f}%")

OC1 seul (pas dans le texte METAR): 19,798
avwx seul (pas dans OC1): 155
count    19798.000000
mean         3.530821
std          2.282500
min          0.500000
25%          2.100000
50%          3.100000
75%          4.600000
max         50.900000
dtype: float64
Sous le seuil des 10kt (5.14 m/s): 82.0%


In [6]:
import pandas as pd

arome = pd.read_csv("../data/cleaned/Arome_clean_final.csv", dtype={"id": str})
arome["datetime"] = pd.to_datetime(arome["datetime"], format="%Y%m%d%H")
arome = arome[(arome["datetime"] >= "2021-01-01") & (arome["datetime"] <= "2025-12-31 23:59")]

merged = pd.read_csv("../data/cleaned/AROME_METAR_merged_2021_2025.csv", dtype={"id": str}, low_memory=False)

arome_counts = arome.groupby("id").size()
merged_counts = merged.groupby("id").size()

coverage = (merged_counts / arome_counts * 100).round(1).sort_values()
print(coverage)

id
60120    13.5
60096    19.0
60250    20.7
60060    30.1
60155    53.1
60285    56.2
60200    58.8
60191    62.1
60280    67.3
60220    80.9
60033    89.1
60107    89.9
60150    90.5
60210    90.8
60265    90.9
60340    91.0
60230    91.1
60101    91.2
60115    91.2
60135    91.2
60141    91.3
60156    91.3
60252    91.3
60338    91.3
60136     NaN
60160     NaN
dtype: float64


In [7]:
import pandas as pd

merged = pd.read_csv("../data/cleaned/AROME_METAR_merged_2021_2025.csv", dtype={"id": str}, low_memory=False)

for station_id in ["60120", "60096", "60250", "60155"]:  # Kenitra, Villa Cisneros, Inezgane, Anfa (référence)
    sub = merged[merged["id"] == station_id]
    hours = pd.to_datetime(sub["date"]).dt.hour.value_counts().sort_index()
    print(f"\n--- {station_id} ---")
    print(hours)


--- 60120 ---
date
0     1424
6     1421
12    1412
18    1386
Name: count, dtype: int64

--- 60096 ---
date
0     403
1     391
2     402
4     395
5     396
6     386
7     396
8     384
10    408
11    404
12    404
13    396
14    399
16    383
17    396
18    393
19    384
20    381
22    395
23    365
Name: count, dtype: int64

--- 60250 ---
date
0      531
3      540
6     1417
9     1428
12    1411
15    1420
18    1410
21     484
Name: count, dtype: int64

--- 60155 ---
date
0        2
3        2
5       23
6     1594
7     1584
8     1591
9     1581
10    1587
11    1575
12    1582
13    1561
14    1573
15    1589
16    1571
17    1559
18    1590
19    1564
20      18
21       2
Name: count, dtype: int64
